# Introdução à Classificação
**Prof. Dr. Eduardo Pena — DACOM/UTFPR**  

**Objetivos desta aula:**
- Entender o conceito de classificação binária
- Explorar dados de doenças cardíacas
- Ajustar modelos de Regressão Logística, Naive Bayes e K-NN
- Interpretar probabilidades, coeficientes e métricas de classificação
- Avaliar qualidade dos modelos através de matriz de confusão e métricas

---

## O que é Classificação?

A **classificação** é uma técnica de aprendizado supervisionado que prediz:
- Uma **variável categórica** (y) - a classe que queremos prever
- Baseada em **variáveis preditoras** (x) - as características dos dados

**Classificação Binária:** duas classes (0/1, Sim/Não, Positivo/Negativo)
**Classificação Multiclasse:** três ou mais classes

### Algoritmos que estudaremos:
1. **Regressão Logística** - modelo probabilístico linear
2. **Naive Bayes** - baseado em probabilidades condicionais
3. **K-NN** - baseado em proximidade/similaridade

## 1. Imports e Configurações

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Algoritmos de classificação
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Métricas de avaliação
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

# Pré-processamento
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Configurações de visualização
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
sns.set_theme(style="whitegrid")

# Para reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Carregamento e Exploração dos Dados

Vamos trabalhar com um dataset sobre **doenças cardíacas**:
- **age**: idade do paciente
- **sex**: sexo (1=masculino, 0=feminino)
- **cp**: tipo de dor no peito (0-3)
- **trestbps**: pressão arterial em repouso
- **chol**: colesterol sérico
- **fbs**: açúcar no sangue em jejum > 120 mg/dl (1=true, 0=false)
- **restecg**: resultados do eletrocardiograma em repouso (0-2)
- **thalach**: frequência cardíaca máxima alcançada
- **exang**: angina induzida por exercício (1=sim, 0=não)
- **oldpeak**: depressão do ST induzida por exercício
- **slope**: inclinação do segmento ST no pico do exercício
- **ca**: número de vasos principais coloridos por fluoroscopia (0-3)
- **thal**: talassemia (0=normal, 1=defeito fixo, 2=defeito reversível)
- **condition**: presença de doença cardíaca (1=sim, 0=não) - **VARIÁVEL TARGET**

In [ ]:

heart = pd.read_csv('heart.csv')


print("Primeiras 5 linhas dos dados:")
heart.head()



In [ ]:
print("\nInformações sobre o dataset:")
print(heart.info())

print("\nEstatísticas:")
print(heart.describe())

print(f"\nDistribuição da variável target 'condition':")
target_counts = heart['condition'].value_counts()
print(target_counts)
print(f"\nProporção de classes:")
print(heart['condition'].value_counts(normalize=True).round(3))

## 3. Visualização e Análise Exploratória

In [ ]:
# Análise de algumas variáveis importantes vs target
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Age vs Condition
heart.boxplot(column='age', by='condition', ax=axes[0,0])
axes[0,0].set_title('Idade por Presença de Doença Cardíaca')
axes[0,0].set_xlabel('Condition (0=Sem doença, 1=Com doença)')
axes[0,0].set_ylabel('Idade')

# Cholesterol vs Condition
heart.boxplot(column='chol', by='condition', ax=axes[0,1])
axes[0,1].set_title('Colesterol por Presença de Doença Cardíaca')
axes[0,1].set_xlabel('Condition (0=Sem doença, 1=Com doença)')
axes[0,1].set_ylabel('Colesterol')

# Max Heart Rate vs Condition
heart.boxplot(column='thalach', by='condition', ax=axes[1,0])
axes[1,0].set_title('Freq. Cardíaca Máxima por Presença de Doença')
axes[1,0].set_xlabel('Condition (0=Sem doença, 1=Com doença)')
axes[1,0].set_ylabel('Frequência Cardíaca Máxima')

# Sex vs Condition (proporções)
crosstab = pd.crosstab(heart['sex'], heart['condition'], normalize='index')
crosstab.plot(kind='bar', ax=axes[1,1], color=['lightcoral', 'lightblue'])
axes[1,1].set_title('Proporção de Doença por Sexo')
axes[1,1].set_xlabel('Sexo (0=Feminino, 1=Masculino)')
axes[1,1].set_ylabel('Proporção')
axes[1,1].legend(['Sem doença', 'Com doença'])
axes[1,1].tick_params(axis='x', rotation=0)

fig.suptitle('Distribuição de Variáveis Importantes vs Doença Cardíaca', fontsize=18)

plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlação
plt.figure(figsize=(12, 10))
correlation_matrix = heart.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de Correlação - Dados de Doenças Cardíacas')
plt.show()

# Correlações com a variável target
print("Correlações com 'condition' (ordenadas):")
correlations = heart.corr()['condition'].sort_values(ascending=False)
print(correlations.round(3))

## 4. Preparação dos Dados

Vamos preparar os dados para os algoritmos de classificação.

In [ ]:
# Separar features (X) e target (y)
X = heart.drop('condition', axis=1)
y = heart['condition']

# Dividir em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Tamanho do conjunto de treino: {X_train.shape[0]} amostras")
print(f"Tamanho do conjunto de teste: {X_test.shape[0]} amostras")

print(f"\nDistribuição das classes no treino:")
print(y_train.value_counts())

print(f"\nDistribuição das classes no teste:")
print(y_test.value_counts())

# Visualizar as features que usaremos
print(f"\nFeatures disponíveis ({len(X.columns)}):")
print(list(X.columns))

## 5. Regressão Logística

A **Regressão Logística** modela a probabilidade de uma classe usando a função logística (sigmoide):

**P(y=1|x) = 1 / (1 + e^(-(β₀ + β₁x₁ + β₂x₂ + ... + βₚxₚ)))**

### Características:
- Produz probabilidades entre 0 e 1
- Coeficientes têm interpretação clara
- Assume relação linear entre features e log-odds
- Robusto e rápido

In [ ]:
# Criar e treinar o modelo de Regressão Logística
logistic_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
logistic_model.fit(X_train, y_train)

# Fazer previsões
y_pred_logistic = logistic_model.predict(X_test)
y_pred_proba_logistic = logistic_model.predict_proba(X_test)[:, 1]  # Probabilidade da classe positiva

print("Regressão Logística - Resultados:")
print("=" * 50)

# Métricas de avaliação
accuracy_logistic = accuracy_score(y_test, y_pred_logistic)
precision_logistic = precision_score(y_test, y_pred_logistic)
recall_logistic = recall_score(y_test, y_pred_logistic)
f1_logistic = f1_score(y_test, y_pred_logistic)
auc_logistic = roc_auc_score(y_test, y_pred_proba_logistic)

print(f"Acurácia: {accuracy_logistic:.3f}")
print(f"Precisão: {precision_logistic:.3f}")
print(f"Recall (Sensibilidade): {recall_logistic:.3f}")
print(f"F1-Score: {f1_logistic:.3f}")
print(f"AUC: {auc_logistic:.3f}")

print(f"\nInterpretação das métricas:")
print(f"Acurácia: {accuracy_logistic*100:.1f}% das previsões estão corretas")
print(f"Precisão: {precision_logistic*100:.1f}% dos casos preditos como doença realmente têm doença")
print(f"Recall: {recall_logistic*100:.1f}% dos casos com doença foram identificados")
print(f"F1-Score: média harmônica entre precisão e recall")
print(f"AUC: {auc_logistic:.3f} - capacidade de discriminação (0.5=aleatório, 1.0=perfeito)")

In [ ]:
# Matriz de Confusão
cm_logistic = confusion_matrix(y_test, y_pred_logistic)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_logistic, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Sem doença', 'Com doença'],
            yticklabels=['Sem doença', 'Com doença'])
plt.title('Matriz de Confusão - Regressão Logística')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.show()

print("Interpretação da Matriz de Confusão:")
print(f"• Verdadeiros Negativos (TN): {cm_logistic[0,0]} - sem doença corretamente identificados")
print(f"• Falsos Positivos (FP): {cm_logistic[0,1]} - sem doença classificados como com doença")
print(f"• Falsos Negativos (FN): {cm_logistic[1,0]} - com doença classificados como sem doença")
print(f"• Verdadeiros Positivos (TP): {cm_logistic[1,1]} - com doença corretamente identificados")

print(f"\nErros críticos:")
print(f"• Falsos Negativos: {cm_logistic[1,0]} pacientes com doença não detectados (mais perigoso)")
print(f"• Falsos Positivos: {cm_logistic[0,1]} pacientes saudáveis classificados como doentes")

In [ ]:
# Análise dos coeficientes da Regressão Logística
feature_names = X.columns
coefficients = logistic_model.coef_[0]
intercept = logistic_model.intercept_[0]

# Criar DataFrame com coeficientes
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients)
}).sort_values('Abs_Coefficient', ascending=False)

print("Coeficientes da Regressão Logística:")
print("=" * 50)
print(f"Intercepto: {intercept:.3f}")
print("\nCoeficientes (ordenados por importância):")
print(coef_df.round(3))

# Visualizar coeficientes
plt.figure(figsize=(12, 8))
plt.barh(range(len(coef_df)), coef_df['Coefficient'], color=['red' if c < 0 else 'blue' for c in coef_df['Coefficient']])
plt.yticks(range(len(coef_df)), coef_df['Feature'])
plt.xlabel('Coeficiente')
plt.title('Coeficientes da Regressão Logística')
plt.axvline(x=0, color='black', linestyle='--', alpha=0.7)
plt.grid(True, alpha=0.3)
plt.show()

print("\nInterpretação dos coeficientes:")
print("• Coeficiente positivo: aumenta chance de doença cardíaca")
print("• Coeficiente negativo: diminui chance de doença cardíaca")
print("• Magnitude: quanto maior o valor absoluto, maior o impacto")

## 6. Naive Bayes

O **Naive Bayes** é baseado no Teorema de Bayes e assume independência entre features:

**P(classe|features) = P(features|classe) × P(classe) / P(features)**

### Características:
- Assume independência entre variáveis (naive)
- Funciona bem com dados pequenos
- Rápido para treinar e predizer
- Probabilístico por natureza
- Funciona bem mesmo quando a suposição de independência é violada

In [ ]:
# Criar e treinar o modelo Naive Bayes
# Usamos GaussianNB (assume distribuição normal das features)
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# Fazer previsões
y_pred_nb = nb_model.predict(X_test)
y_pred_proba_nb = nb_model.predict_proba(X_test)[:, 1]

print("Naive Bayes - Resultados:")
print("=" * 50)

# Métricas de avaliação
accuracy_nb = accuracy_score(y_test, y_pred_nb)
precision_nb = precision_score(y_test, y_pred_nb)
recall_nb = recall_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)
auc_nb = roc_auc_score(y_test, y_pred_proba_nb)

print(f"Acurácia: {accuracy_nb:.3f}")
print(f"Precisão: {precision_nb:.3f}")
print(f"Recall (Sensibilidade): {recall_nb:.3f}")
print(f"F1-Score: {f1_nb:.3f}")
print(f"AUC: {auc_nb:.3f}")

print(f"\nComo o Naive Bayes funciona:")
print(f"Calcula P(sem doença|features) e P(com doença|features)")
print(f"Assume que todas as features são independentes")
print(f"Escolhe a classe com maior probabilidade")
print(f"Muito eficiente e funciona bem mesmo com poucos dados")

In [ ]:
# Matriz de Confusão para Naive Bayes
cm_nb = confusion_matrix(y_test, y_pred_nb)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Sem doença', 'Com doença'],
            yticklabels=['Sem doença', 'Com doença'])
plt.title('Matriz de Confusão - Naive Bayes')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.show()

print("Comparação com Regressão Logística:")
print(f"• Falsos Negativos - Logística: {cm_logistic[1,0]} | Naive Bayes: {cm_nb[1,0]}")
print(f"• Falsos Positivos - Logística: {cm_logistic[0,1]} | Naive Bayes: {cm_nb[0,1]}")

In [ ]:
# Análise das probabilidades das classes (priors)
class_priors = nb_model.class_prior_

print("Probabilidades a priori das classes (Naive Bayes):")
print(f"P(sem doença) = {class_priors[0]:.3f}")
print(f"P(com doença) = {class_priors[1]:.3f}")

print(f"\nEstas probabilidades correspondem às proporções no conjunto de treino:")
print(y_train.value_counts(normalize=True).sort_index().round(3))

# Analisar como o modelo vê algumas features importantes
print(f"\nMédia das features por classe (dados de treino):")
feature_means = X_train.groupby(y_train).mean()
print(feature_means.round(2))

# Mostrar as diferenças mais significativas
feature_diff = (feature_means.loc[1] - feature_means.loc[0]).abs().sort_values(ascending=False)
print(f"\nMaiores diferenças entre classes:")
print(feature_diff.head().round(3))

## 7. K-NN para Classificação

O **K-NN** para classificação funciona similar à regressão, mas:
- Encontra os K vizinhos mais próximos
- **Voto majoritário**: a classe mais comum entre os K vizinhos
- Pode retornar probabilidades baseadas na proporção de votos

### Diferenças em relação ao K-NN para regressão:
- **Saída**: classe categórica ao invés de valor numérico
- **Decisão**: voto majoritário ao invés de média
- **Probabilidades**: proporção de vizinhos de cada classe

In [ ]:
# K-NN é sensível à escala, vamos usar um pipeline com normalização
print("K-NN para Classificação:")
print("=" * 50)

# Criar pipeline que padroniza os dados e aplica K-NN
knn_model = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

# Treinar o modelo
knn_model.fit(X_train, y_train)

# Fazer previsões
y_pred_knn = knn_model.predict(X_test)
y_pred_proba_knn = knn_model.predict_proba(X_test)[:, 1]

# Métricas de avaliação
accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn)
recall_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)
auc_knn = roc_auc_score(y_test, y_pred_proba_knn)

print(f"K-NN (K=5):")
print(f"Acurácia: {accuracy_knn:.3f}")
print(f"Precisão: {precision_knn:.3f}")
print(f"Recall (Sensibilidade): {recall_knn:.3f}")
print(f"F1-Score: {f1_knn:.3f}")
print(f"AUC: {auc_knn:.3f}")

print(f"\nComo o K-NN para classificação funciona:")
print(f"Para cada ponto de teste, encontra os 5 vizinhos mais próximos")
print(f"Conta quantos vizinhos são de cada classe")
print(f"Atribui a classe majoritária (voto majoritário)")
print(f"Probabilidade = proporção de vizinhos de cada classe")

### Testando diferentes valores de K para classificação

In [ ]:
# Testar diferentes valores de K para classificação
k_values = [1, 3, 5, 7, 9, 11, 15, 20]
results_classification = []

for k in k_values:
    # Criar modelo K-NN com k vizinhos
    knn_k = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    
    # Treinar e avaliar
    knn_k.fit(X_train, y_train)
    y_pred_k = knn_k.predict(X_test)
    y_pred_proba_k = knn_k.predict_proba(X_test)[:, 1]
    
    accuracy_k = accuracy_score(y_test, y_pred_k)
    precision_k = precision_score(y_test, y_pred_k)
    recall_k = recall_score(y_test, y_pred_k)
    f1_k = f1_score(y_test, y_pred_k)
    auc_k = roc_auc_score(y_test, y_pred_proba_k)
    
    results_classification.append({
        'K': k,
        'Accuracy': accuracy_k,
        'Precision': precision_k,
        'Recall': recall_k,
        'F1': f1_k,
        'AUC': auc_k
    })

# Converter para DataFrame
results_knn_df = pd.DataFrame(results_classification)
print("Desempenho do K-NN com diferentes valores de K:")
print(results_knn_df.round(3))

# Encontrar o melhor K baseado no F1-Score
best_k_idx = results_knn_df['F1'].idxmax()
best_k_class = results_knn_df.loc[best_k_idx, 'K']
best_f1 = results_knn_df.loc[best_k_idx, 'F1']
print(f"\nMelhor valor de K: {best_k_class} (F1-Score = {best_f1:.3f})")

In [ ]:
# Visualizar como K afeta as métricas
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Accuracy vs K
axes[0,0].plot(results_knn_df['K'], results_knn_df['Accuracy'], 'bo-', linewidth=2, markersize=8)
axes[0,0].set_xlabel('Valor de K')
axes[0,0].set_ylabel('Accuracy')
axes[0,0].set_title('Accuracy vs K')
axes[0,0].grid(True, alpha=0.3)

# Precision vs K
axes[0,1].plot(results_knn_df['K'], results_knn_df['Precision'], 'go-', linewidth=2, markersize=8)
axes[0,1].set_xlabel('Valor de K')
axes[0,1].set_ylabel('Precision')
axes[0,1].set_title('Precision vs K')
axes[0,1].grid(True, alpha=0.3)

# Recall vs K
axes[1,0].plot(results_knn_df['K'], results_knn_df['Recall'], 'ro-', linewidth=2, markersize=8)
axes[1,0].set_xlabel('Valor de K')
axes[1,0].set_ylabel('Recall')
axes[1,0].set_title('Recall vs K')
axes[1,0].grid(True, alpha=0.3)

# F1-Score vs K
axes[1,1].plot(results_knn_df['K'], results_knn_df['F1'], 'mo-', linewidth=2, markersize=8)
axes[1,1].set_xlabel('Valor de K')
axes[1,1].set_ylabel('F1-Score')
axes[1,1].set_title('F1-Score vs K')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Interpretação dos gráficos:")
print("• K ímpar evita empates na votação")
print("• K muito baixo: sensível a ruído (overfitting)")
print("• K muito alto: pode perder padrões locais (underfitting)")
print("• Trade-off entre precisão e recall varia com K")

## 8. Comparação dos Três Algoritmos

Agora vamos comparar diretamente Regressão Logística, Naive Bayes e K-NN:

In [ ]:
# Treinar o melhor modelo K-NN
best_knn_final = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=int(best_k_class)))
])
best_knn_final.fit(X_train, y_train)
y_pred_knn_final = best_knn_final.predict(X_test)
y_pred_proba_knn_final = best_knn_final.predict_proba(X_test)[:, 1]

# Métricas finais do melhor K-NN
accuracy_knn_final = accuracy_score(y_test, y_pred_knn_final)
precision_knn_final = precision_score(y_test, y_pred_knn_final)
recall_knn_final = recall_score(y_test, y_pred_knn_final)
f1_knn_final = f1_score(y_test, y_pred_knn_final)
auc_knn_final = roc_auc_score(y_test, y_pred_proba_knn_final)

# Comparação final
comparison_final = pd.DataFrame({
    'Modelo': [
        'Regressão Logística',
        'Naive Bayes',
        f'K-NN (K={int(best_k_class)})'
    ],
    'Accuracy': [accuracy_logistic, accuracy_nb, accuracy_knn_final],
    'Precision': [precision_logistic, precision_nb, precision_knn_final],
    'Recall': [recall_logistic, recall_nb, recall_knn_final],
    'F1-Score': [f1_logistic, f1_nb, f1_knn_final],
    'AUC': [auc_logistic, auc_nb, auc_knn_final]
})

print("COMPARAÇÃO FINAL DOS MODELOS:")
print("=" * 60)
print(comparison_final.round(3))

# Identificar o melhor modelo para cada métrica
print(f"\nMelhor modelo por métrica:")
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']:
    best_idx = comparison_final[metric].idxmax()
    best_model = comparison_final.loc[best_idx, 'Modelo']
    best_value = comparison_final.loc[best_idx, metric]
    print(f"• {metric}: {best_model} ({best_value:.3f})")

In [ ]:
# Visualização comparativa das métricas
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(15, 8))

rects1 = ax.bar(x - width, comparison_final.iloc[0, 1:].values, width, label='Regressão Logística', color='blue', alpha=0.7)
rects2 = ax.bar(x, comparison_final.iloc[1, 1:].values, width, label='Naive Bayes', color='green', alpha=0.7)
rects3 = ax.bar(x + width, comparison_final.iloc[2, 1:].values, width, label=f'K-NN (K={int(best_k_class)})', color='red', alpha=0.7)

ax.set_xlabel('Métricas')
ax.set_ylabel('Valores')
ax.set_title('Comparação de Desempenho dos Algoritmos de Classificação')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.grid(True, alpha=0.3)

# Adicionar valores nas barras
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

autolabel(rects1)
autolabel(rects2)
autolabel(rects3)

plt.tight_layout()
plt.show()

## 9. Fazendo Previsões com os Modelos

Vamos usar nossos modelos para fazer previsões em novos casos práticos.

In [ ]:
# Criar exemplos de novos pacientes
novos_pacientes = pd.DataFrame({
    'age': [45, 65, 55],
    'sex': [1, 0, 1],  # 1=masculino, 0=feminino
    'cp': [2, 0, 3],   # tipo de dor no peito
    'trestbps': [120, 160, 140],  # pressão arterial
    'chol': [200, 280, 250],      # colesterol
    'fbs': [0, 1, 0],   # açúcar no sangue
    'restecg': [0, 2, 1],  # ECG
    'thalach': [170, 120, 150],   # freq cardíaca máxima
    'exang': [0, 1, 0],   # angina por exercício
    'oldpeak': [0.5, 2.5, 1.0],  # depressão ST
    'slope': [1, 2, 1],   # inclinação ST
    'ca': [0, 2, 1],      # vasos principais
    'thal': [2, 0, 1]     # talassemia
})

print("Novos pacientes para diagnóstico:")
print("=" * 50)

for i in range(len(novos_pacientes)):
    print(f"\nPaciente {i+1}:")
    print(f"  Idade: {novos_pacientes.iloc[i]['age']} anos")
    print(f"  Sexo: {'Masculino' if novos_pacientes.iloc[i]['sex'] else 'Feminino'}")
    print(f"  Pressão arterial: {novos_pacientes.iloc[i]['trestbps']} mmHg")
    print(f"  Colesterol: {novos_pacientes.iloc[i]['chol']} mg/dl")
    print(f"  Freq. cardíaca máx: {novos_pacientes.iloc[i]['thalach']} bpm")

In [ ]:
# Fazer previsões com todos os modelos
pred_logistic = logistic_model.predict(novos_pacientes)
proba_logistic = logistic_model.predict_proba(novos_pacientes)[:, 1]

pred_nb = nb_model.predict(novos_pacientes)
proba_nb = nb_model.predict_proba(novos_pacientes)[:, 1]

pred_knn = best_knn_final.predict(novos_pacientes)
proba_knn = best_knn_final.predict_proba(novos_pacientes)[:, 1]

print("PREVISÕES DOS MODELOS:")
print("=" * 60)

for i in range(len(novos_pacientes)):
    print(f"\nPaciente {i+1}:")
    print(f"  Regressão Logística: {'COM doença' if pred_logistic[i] else 'SEM doença'} (prob: {proba_logistic[i]:.1%})")
    print(f"  Naive Bayes:         {'COM doença' if pred_nb[i] else 'SEM doença'} (prob: {proba_nb[i]:.1%})")
    print(f"  K-NN:                {'COM doença' if pred_knn[i] else 'SEM doença'} (prob: {proba_knn[i]:.1%})")
    
    # Consenso dos modelos
    total_votes = pred_logistic[i] + pred_nb[i] + pred_knn[i]
    avg_prob = (proba_logistic[i] + proba_nb[i] + proba_knn[i]) / 3
    
    if total_votes >= 2:
        consensus = "COM doença (MAIORIA)"
    elif total_votes == 1:
        consensus = "DISCORDÂNCIA entre modelos"
    else:
        consensus = "SEM doença (MAIORIA)"
    
    print(f"  CONSENSO: {consensus} (prob média: {avg_prob:.1%})")

## 10. Importância das Features

Vamos analisar quais características são mais importantes para cada algoritmo.

In [ ]:
# Análise de importância das features

# 1. Regressão Logística - coeficientes (já vimos antes)
print("IMPORTÂNCIA DAS FEATURES:")
print("=" * 50)

print("\n1. REGRESSÃO LOGÍSTICA (valor absoluto dos coeficientes):")
feature_importance_lr = pd.DataFrame({
    'Feature': X.columns,
    'Importance': np.abs(logistic_model.coef_[0])
}).sort_values('Importance', ascending=False)

print(feature_importance_lr.round(3))

# 2. Para Naive Bayes, vamos analisar as diferenças entre médias das classes
print("\n2. NAIVE BAYES (diferença entre médias das classes):")
class_means = X_train.groupby(y_train).mean()
feature_importance_nb = pd.DataFrame({
    'Feature': X.columns,
    'Importance': np.abs(class_means.loc[1] - class_means.loc[0])
}).sort_values('Importance', ascending=False)

print(feature_importance_nb.round(3))

# 3. Para K-NN, vamos usar uma abordagem de permutação simples
print("\n3. K-NN - Top 5 features mais importantes (baseado em análise anterior):")
print("Note: K-NN não fornece importância direta das features")
print("As features mais discriminativas baseadas na correlação:")
importance_corr = heart.corr()['condition'].abs().sort_values(ascending=False)[1:6]  # Excluir a própria variável
print(importance_corr.round(3))

In [ ]:
# Visualização da importância das features
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Regressão Logística
top_10_lr = feature_importance_lr.head(10)
axes[0].barh(range(len(top_10_lr)), top_10_lr['Importance'], color='blue', alpha=0.7)
axes[0].set_yticks(range(len(top_10_lr)))
axes[0].set_yticklabels(top_10_lr['Feature'])
axes[0].set_xlabel('Importância (|Coeficiente|)')
axes[0].set_title('Importância das Features - Regressão Logística')
axes[0].grid(True, alpha=0.3)

# Naive Bayes
top_10_nb = feature_importance_nb.head(10)
axes[1].barh(range(len(top_10_nb)), top_10_nb['Importance'], color='green', alpha=0.7)
axes[1].set_yticks(range(len(top_10_nb)))
axes[1].set_yticklabels(top_10_nb['Feature'])
axes[1].set_xlabel('Importância (Diferença entre médias)')
axes[1].set_title('Importância das Features - Naive Bayes')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Features mais importantes em comum
print("\nFeatures mais importantes (comum aos dois algoritmos):")
common_important = set(top_10_lr['Feature'].head(5)) & set(top_10_nb['Feature'].head(5))
print(f"Top 5 em comum: {common_important}")